# Sparse labelling of waterhole surface state

Paints class labels onto the **GeoTIFF's own grid**. The display panels are rendered live
from the raster and the mask is written back at exactly that size and transform — nothing
is resampled, because a resampled label is a wrong label.

**Sparse by design.** You are not filling in whole tiles. Paint a few confident patches per
class and move on. At 10 m most basin margins are mixed pixels, and leaving them as class 0
is the correct answer, not an unfinished job.

**Month stepping keeps your work.** Left/right arrows walk through the same site's months so
you can see what a pixel did before and after, without losing any label buffer. Nothing
touches disk until you press `ctrl+s`.

The tool opens in a **separate window**, not inline — painting needs a responsive canvas.

## Setup

`%matplotlib qt` opens a native Qt window. If Qt misbehaves on your machine, `%matplotlib
macosx` also works. Inline backends will not work: they cannot deliver mouse-drag events.

In [22]:
%matplotlib qt

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_config
import wh_footprint
import wh_inventory
import wh_label

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)

print("config", cfg.source_path.name, "hash", cfg.hash)
print(f"{len(manifest):,} chips, {manifest['site_id'].nunique()} sites")
print("\nclass scheme:")
for definition in cfg.classes:
    print(f"  [{definition.key}] {definition.id} {definition.name:24s} {definition.colour}")

config waterhole_seg_config.yaml hash afde19a3ff27
15,708 chips, 187 sites

class scheme:
  [0] 0 unlabelled               #00000000
  [1] 1 open_water               #1f4ea1
  [2] 2 turbid_water             #8c6d3f
  [3] 3 aquatic_vegetation       #2e8b57
  [4] 4 mud                      #a0522d
  [5] 5 dry_bare                 #d9b382
  [6] 6 surrounding_vegetation   #7f9f5a


## Parameters

Display and brush settings, here rather than in the YAML so they are visible while you work.

The **class scheme itself stays in `waterhole_seg_config.yaml`** — training and prediction
have to agree with it, and every saved mask records its `scheme_version`. Change classes
there, not here.

In [23]:
PARAMS = wh_label.LabelParams(
    # Panels in reading order across the grid: with panel_rows=2 the first
    # three fill the top row and the rest the bottom. All share pan and zoom,
    # including across rows.
    # NDMI is here because it responds to water UNDER a canopy, which is the case
    # MNDWI gets wrong when sedges cover a waterhole.
    panels=("rgb", "mndwi", "alphaearth", "ndvi", "ndti", "ndmi"),
    panel_rows=2,          # 1 for a single row; 2 uses the window far better at 6 panels

    # The AlphaEarth panel: 64 learned dimensions rendered as one false-colour
    # image. It is ANNUAL and static, so it does not change as you step through
    # months — it is structural context, not a reading of this month's surface.
    #   "bands" -> the three named below straight to R, G, B
    #   "pca"   -> first three principal components of all 64
    # A60/A24/A63 ranked highest on permutation importance.
    alphaearth_mode="bands",
    alphaearth_bands=("A28", "A32", "A63"),
    alphaearth_year=2025,

    rgb_bands=("B4", "B3", "B2"),
    rgb_max_reflectance=0.30,
    rgb_gamma=0.85,

    # Tight display ranges. Contrast lands on the scene rather than being spent
    # on [-1, 1] values that never occur.
    display_ranges={
        "mndwi": (-0.8, 0.6),
        "ndwi": (-0.8, 0.6),
        "ndvi": (-0.1, 0.9),
        "ndti": (-0.4, 0.4),
        "ndmi": (-0.6, 0.6),
    },

    brush_radius_px=2,
    max_brush_radius_px=20,
    undo_depth=200,
    label_alpha=0.55,
    initial_zoom_half_width_px=50,
    labeller_name="scott",
)

print(wh_label.KEY_HELP)


  PAINTING
    0-6              select class (0 = unlabelled)
    b / e            brush / eraser
    g                polygon fill (drag to enclose)
    [ / ]            brush smaller / larger
    z    or cmd+z    undo
    Z    or cmd+y    redo

  NAVIGATION
    left / right     previous / next MONTH of this waterhole
    c                return to this waterhole's starting month
    p / n            previous / next WATERHOLE
    s                skip to the next waterhole

  ACTIONS
    w    or cmd+s    save this month
    W                save EVERY month with unsaved labels
    h                hide/show labels
    f                hide/show basin footprint
    v                open/close the pre-rendered PNG (separate window)
    q                close

  On macOS cmd and ctrl are interchangeable for every binding above.
  Matplotlib's own shortcuts are disabled inside this window (they collide
  with p, s, f, h, g, v and the arrow keys) and restored when it closes.

  SAVING IS 

## Build the work queue

**One row per waterhole**, not per site-month. That is the unit of work: you pick a
waterhole, look through its history with the arrow keys, label the months that are
informative, and move on with `n`.

It also matches how the results get validated. With grouped-by-site cross-validation the
effective sample size is the number of labelled **sites**, not pixels or months — so
breadth across waterholes is what buys statistical power, and depth within one buys much
less than it feels like it does.

`start_month` chooses where each site opens. Either way the choice is made among months
that pass the quality filters, so a site never opens on a chip that is mostly cloud —
labelling a median built from one cloudy scene teaches the classifier noise. Every month
stays browsable with the arrow keys regardless of where you start.

In [24]:
SITES = None       # e.g. ["025", "075", "000"], or None for every site

queue = wh_label.build_queue(
    manifest,
    sites=SITES,
    max_gap_fraction=0.20,           # a month must be <20% unobserved to be a start month
    min_mean_obs=2.0,                # ...and rest on at least 2 scenes on average

    # Which month each waterhole opens on:
    #   "first"          earliest month of the record — start at the beginning
    #                    and work forward, keeping a site's history in order
    #   "best_observed"  clearest month of the late dry season, when the basin
    #                    floor and pugged margin are easiest to read
    start_month="first",
    start_month_preference=(9, 10, 8, 7),   # only used by "best_observed"
)

print(f"{len(queue)} waterholes queued")
queue.head(10)

187 waterholes queued


,site_id,start_year_month,start_tif_path,start_month_index,n_good_months,mean_n_obs
0,000,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,74,4.401452
1,001,2019-02,/Users/scottforrest/Library/CloudStorage/OneDr...,24229,73,4.399895
2,002,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,74,4.373291
3,003,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,72,4.375262
4,004,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,72,4.427098
5,005,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,72,4.561220
6,006,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,76,4.334271
7,007,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,75,4.268069
8,008,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,72,4.441739
9,009,2019-01,/Users/scottforrest/Library/CloudStorage/OneDr...,24228,72,4.434867


### Optional: load basin footprints

If you have run `footprint_estimation.ipynb`, the footprint outline is drawn on every panel
(toggle with `f`). It is a guide for where the basin is, not a constraint on where you can
paint — label what you see, not what the footprint says.

In [25]:
footprints = {}
for site_id in queue["site_id"].unique():
    try:
        footprints[site_id] = wh_footprint.load_mask(cfg, site_id)
    except FileNotFoundError:
        pass

print(f"loaded {len(footprints)} footprint(s) of {queue['site_id'].nunique()} queued sites")

loaded 176 footprint(s) of 187 queued sites


## Launch

Opens the labelling window. Keep this notebook running while you work.

**Saving is automatic.** A month with unsaved labels is written whenever you navigate away
from it, and everything outstanding is written when the window closes. An empty mask is
never written, so autosave cannot litter the labels directory. `W` saves everything
outstanding at any time, and `w` (or `cmd+s`) saves just the current month.

### How to label well

1. **Step through the months first** (left/right) before painting anything. Ambiguous
   Octobers become obvious once you have seen the same pixels in February.
2. **Watch the readout** in the bottom right — band values, every index, and `n_obs` for
   the pixel under the cursor. If `n_obs` is 1, be sceptical of what you are seeing.
3. **Paint conservatively.** A few tens of confident pixels per class per month is plenty.
   Leave mixed basin margins as class 0; at 10 m that is the honest answer.
4. **Breadth beats depth.** Two or three months per waterhole across many waterholes is
   worth more than every month of a few, because cross-validation groups by site.
5. **Spend your effort on the hard classes.** Open water and surrounding vegetation will be
   generated automatically by the pseudo-labeller. Aquatic vegetation, wet mud and pugged
   margin are what actually need your judgement.

In [26]:
labeller = wh_label.launch(queue, manifest, cfg, PARAMS, footprints=footprints)

## Session progress

Run this after labelling to see what has been written to disk.

In [27]:
label_dir = cfg.paths["labels"]
sidecars = sorted(label_dir.glob("*_labels.json")) if label_dir.exists() else []

import json as _json
records = []
for path in sidecars:
    meta = _json.loads(path.read_text())
    row = {
        "site_id": meta["site_id"],
        "year_month": meta["year_month"],
        "labeller": meta["labeller"],
        "source": meta["source"],
        "n_labelled": meta["n_labelled"],
    }
    row.update(meta["pixel_counts"])
    records.append(row)

if records:
    progress = pd.DataFrame(records).sort_values(["site_id", "year_month"])
    print(f"{len(progress)} labelled tiles across {progress['site_id'].nunique()} sites")
    print(f"{progress['n_labelled'].sum():,} labelled pixels total\n")
    class_names = [d.name for d in cfg.classes if not d.ignore]
    print("pixels per class:")
    print(progress[class_names].sum().sort_values(ascending=False).to_string())
    display(progress)
else:
    print("no labels saved yet")

313 labelled tiles across 27 sites
95,429 labelled pixels total

pixels per class:
surrounding_vegetation    41013
dry_bare                  28557
aquatic_vegetation        12647
turbid_water               5855
mud                        4457
open_water                 2900


,site_id,year_month,labeller,source,n_labelled,unlabelled,open_water,turbid_water,aquatic_vegetation,mud,dry_bare,surrounding_vegetation
0,000,2022-05,scott,manual,13,22637,0,0,0,13,0,0
1,000,2022-06,scott,manual,13,22637,0,0,0,13,0,0
2,000,2022-08,scott,manual,13,22637,0,0,0,13,0,0
3,000,2023-08,scott,manual,13,22637,0,0,0,13,0,0
4,000,2023-09,scott,manual,13,22637,0,0,0,13,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
308,096,2022-03,scott,manual,81,22569,0,81,0,0,0,0
309,096,2022-04,scott,manual,81,22569,0,81,0,0,0,0
310,096,2022-05,scott,manual,122,22528,0,122,0,0,0,0
311,096,2022-06,scott,manual,134,22516,0,134,0,0,0,0


### Class balance

Surrounding vegetation will vastly outnumber everything else, which is expected and handled
by class weighting at training time. What matters here is whether the *hard* classes —
aquatic vegetation, mud, dry bare — have enough labelled **sites** behind them, not enough
pixels. A class present at only one site cannot be cross-validated.

In [28]:
if records:
    class_names = [d.name for d in cfg.classes if not d.ignore]
    per_class = pd.DataFrame({
        "pixels": progress[class_names].sum(),
        "tiles": (progress[class_names] > 0).sum(),
        "sites": [
            progress.loc[progress[name] > 0, "site_id"].nunique()
            for name in class_names
        ],
    }).sort_values("sites")
    display(per_class)

    thin = per_class[per_class["sites"] < 3]
    if not thin.empty:
        print("\nclasses labelled at fewer than 3 sites — not yet cross-validatable:")
        print(", ".join(thin.index))
else:
    print("no labels saved yet")

,pixels,tiles,sites
open_water,2900,6,5
turbid_water,5855,49,8
surrounding_vegetation,41013,23,9
mud,4457,127,19
aquatic_vegetation,12647,59,20
dry_bare,28557,78,20


---

### If chips fail to open

The tiles sit on a OneDrive share with Files On-Demand, which dehydrates files to
placeholders. A dehydrated chip reports a normal size but cannot be opened, surfacing as
`errno 60` or *"not recognized as a supported file format"*. Retrying does not help.

Fix: right-click `cookie-cutting` in Finder → **Always Keep on This Device**, or move
`images_tif_v2` off OneDrive. `wh_tiles.read_tile` detects this and says so explicitly.